In [ ]:
!pip install -q openai datasets httpx beautifulsoup4

In [1]:
import asyncio
import hashlib
import json
import os
import random
import re
import tempfile
import time
import warnings
from collections.abc import Awaitable, Callable
from dataclasses import dataclass, field
from pathlib import Path
from urllib.parse import unquote, urlparse

import httpx
from bs4 import BeautifulSoup
from datasets import load_dataset
from openai import AsyncOpenAI

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = ""
os.environ["SERPER_API_KEY"] = ""

client = AsyncOpenAI()
MODEL = "gpt-5-mini"
SEED = 0
NUM_QUESTIONS = 12
# research budget: rounds of (plan -> parallel workers -> gap check)
MAX_ROUNDS = 3
MAX_SUBQUESTIONS = 4
WORKER_STEPS = 3
READS_PER_STEP = 2
# the single-loop baseline gets the same number of search steps as one deep round
BASELINE_STEPS = 6
CACHE_DIR = Path(tempfile.gettempdir()) / "decodeai_deep_research"
CACHE_DIR.mkdir(exist_ok=True)
# USD per million tokens, gpt-5-mini list price
PRICE_IN, PRICE_OUT = 0.25, 2.00

# Deep Research Agent

A deep research agent answers a question from the live web. It searches, reads pages, writes
down evidence with verbatim quotes, and composes an answer whose claims cite that evidence.
The hard questions are **multi-hop**: "the first name of the 15th First Lady's mother plus
the maiden name of the second assassinated president's mother" needs four facts, and each
fact can be found only after the entity behind it has been identified.

**Why a single search loop is shallow.** The simplest agent runs one loop: decide the next
query, search, read, repeat until it feels done. It has three structural weaknesses:

1. **Serial and single-threaded.** One query per step, and every page it reads lands in the
   same context, whichever hop it serves.
2. **Unverified assumptions.** Each step needs only *the single most useful missing fact*, so
   the model fills the gaps from memory ("the second assassinated president is Garfield")
   and never searches them. A wrong assumption is never checked.
3. **No completeness check.** It stops when the notes look plausible. Nothing asks what is
   still missing, or which notes contradict each other.

**What production systems do instead** (OpenAI deep research, Anthropic's multi-agent
research system, open_deep_research): an orchestrator **plans** the question into
sub-questions, runs a **worker per sub-question in parallel** (each with its own small
context), then **reflects** on the combined findings. When something is missing or
conflicting, it writes follow-up questions with the newly found entities filled in and runs
another round.

```
question
   |
   v
 plan: atomic sub-questions (verify every assumption)
   |
   +--> worker 1: search -> read -> notes -> sub-answer     \
   +--> worker 2: search -> read -> notes -> sub-answer      >  in parallel
   +--> worker 3: ...                                       /
   |
   v
 gap check: missing facts? conflicts? unverified steps? --yes--> follow-up questions --+
   | no (or budget spent)                                                              |
   v                                              next round, entities filled in <-----+
 cited answer from notes -> citation precision
```

| Component | Built here | Why it matters |
|---|---|---|
| Search + fetch | Google via Serper, Wikipedia API, caching | real, reproducible web evidence |
| Evidence notes | claim + quote verified on the page | the answer can cite only checked text |
| Planner | atomic, assumption-free sub-questions | every hop gets searched, not assumed |
| Workers | one short loop per sub-question, concurrent | wall-clock stays flat as hops grow |
| Gap check | missing / conflicting / unverified -> follow-ups | recovers when the first plan was wrong |
| Writer + checker | cited report, citation precision | measures whether citations support claims |

---

# 1) Benchmark and Metering

## 1.1 FRAMES

**FRAMES** (Krishna et al., 2024) has 824 questions written so that the answer needs facts
from 2 to 15 Wikipedia articles, with numerical, temporal and multi-constraint reasoning.
Answers are free-form ("Jane Ballou", "37 years"), so a model grades each prediction against
the gold answer, the protocol FRAMES itself uses.

In [2]:
frames = load_dataset("google/frames-benchmark", split="test")


def frames_item(index: int) -> dict:
    """Return one FRAMES row as id, question and gold answer."""
    row = frames[index]
    return {"id": index, "question": row["Prompt"].strip(), "gold": row["Answer"]}


EXAMPLE = frames_item(0)
QUESTIONS = [
    frames_item(i)
    for i in random.Random(SEED).sample(range(1, len(frames)), NUM_QUESTIONS)
]
print("Q:", EXAMPLE["question"])
print("A:", EXAMPLE["gold"])

Q: If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name?
A: Jane Ballou


## 1.2 Model Calls and Cost

Every model call returns strict JSON (structured outputs) and adds its token usage to a
`Meter`, so each method's searches, page reads, dollars and seconds can be compared. A
semaphore caps concurrent model calls, since many workers across many questions run at once.

In [3]:
@dataclass
class Meter:
    """Usage counters for one research run."""

    searches: int = 0
    fetches: int = 0
    input_tokens: int = 0
    output_tokens: int = 0

    @property
    def dollars(self) -> float:
        """Cost of the metered tokens at list price."""
        return (self.input_tokens * PRICE_IN + self.output_tokens * PRICE_OUT) / 1e6


def obj(**properties: dict) -> dict:
    """Strict JSON schema object: every property required, nothing extra."""
    return {
        "type": "object",
        "properties": properties,
        "required": list(properties),
        "additionalProperties": False,
    }


STR, BOOL = {"type": "string"}, {"type": "boolean"}
INT = {"type": "integer"}
STR_LIST = {"type": "array", "items": STR}
LLM_SLOTS = asyncio.Semaphore(32)

In [4]:
async def llm(
    meter: Meter, system: str, user: str, schema: dict, effort: str = "low"
) -> dict:
    """One structured-output model call; usage is added to `meter`."""
    async with LLM_SLOTS:
        response = await client.chat.completions.create(
            model=MODEL,
            reasoning_effort=effort,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {"name": "reply", "strict": True, "schema": schema},
            },
        )
    meter.input_tokens += response.usage.prompt_tokens
    meter.output_tokens += response.usage.completion_tokens
    return json.loads(response.choices[0].message.content)


meter = Meter()
print(await llm(meter, "Answer briefly.", "Capital of Australia?", obj(answer=STR)))
print(meter, f"${meter.dollars:.5f}")

{'answer': 'Canberra.'}
Meter(searches=0, fetches=0, input_tokens=41, output_tokens=19) $0.00005


---

# 2) Tools: Search, Fetch, Evidence Notes

## 2.1 Search and Fetch

Search is Google through the Serper API. Wikipedia pages are fetched as plain text through
the MediaWiki API (Wikipedia refuses anonymous scrapers and prefers its API), and other
pages are fetched with `httpx` and stripped of scripts and navigation. Both are cached on
disk, so reruns are cheap and reproducible. Results from Hugging Face, GitHub, arXiv and
Kaggle are dropped, because those sites host the FRAMES questions with their answers.

In [5]:
BLOCKED_DOMAINS = ("huggingface.co", "github.com", "arxiv.org", "kaggle.com")
MAX_PAGE_CHARS = 20_000
USER_AGENT = "DecodeAI-notebook/1.0 (https://github.com/decodeai; educational demo)"
HTTP = httpx.AsyncClient(
    timeout=20, follow_redirects=True, headers={"User-Agent": USER_AGENT}
)


def cached(kind: str, key: str) -> Path:
    """Cache file for a search query or a page URL."""
    return CACHE_DIR / f"{kind}_{hashlib.sha256(key.encode()).hexdigest()[:24]}.json"


async def search(query: str, meter: Meter) -> list[dict]:
    """Top Google results for `query`, minus benchmark mirrors."""
    meter.searches += 1
    path = cached("search", query)
    if not path.exists():
        response = await HTTP.post(
            "https://google.serper.dev/search",
            headers={"X-API-KEY": os.environ["SERPER_API_KEY"]},
            json={"q": query, "num": 8},
        )
        path.write_text(json.dumps(response.json().get("organic", [])))
    return [
        {"title": r["title"], "url": r["link"], "snippet": r.get("snippet", "")}
        for r in json.loads(path.read_text())
        if not urlparse(r["link"]).netloc.endswith(BLOCKED_DOMAINS)
    ]


meter = Meter()
results = await search("15th First Lady of the United States", meter)
for r in results[:3]:
    print(r["url"], "|", r["snippet"][:70])

https://en.wikipedia.org/wiki/Harriet_Lane | Harriet Rebecca Lane Johnston (née Lane; May 9, 1830 – July 3, 1903) a
https://obamawhitehouse.archives.gov/1600/first-ladies/harrietlane | Harriet Rebecca Lane Johnston acted as First Lady of the United States
https://www.whitehousehistory.org/bios/harriet-lane | Born to parents Elliott T. Lane and Jane Buchanan Lane on May 9, 1830,


In [6]:
async def fetch_text(url: str) -> str:
    """Plain text of a page; Wikipedia articles via the API."""
    parsed = urlparse(url)
    if parsed.netloc.endswith("wikipedia.org") and parsed.path.startswith("/wiki/"):
        response = await HTTP.get(
            f"https://{parsed.netloc}/w/api.php",
            params={
                "action": "query",
                "prop": "extracts",
                "explaintext": 1,
                "format": "json",
                "redirects": 1,
                "titles": unquote(parsed.path[len("/wiki/") :]),
            },
        )
        pages = response.json()["query"]["pages"]
        return next(iter(pages.values())).get("extract", "")
    response = await HTTP.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    return (soup.body or soup).get_text(" ")


async def fetch(url: str, meter: Meter) -> str:
    """Cached, whitespace-collapsed page text, truncated to MAX_PAGE_CHARS."""
    meter.fetches += 1
    path = cached("page", url)
    if not path.exists():
        try:
            text = re.sub(r"\s+", " ", await fetch_text(url)).strip()
        except (httpx.HTTPError, ValueError, KeyError):
            text = ""
        path.write_text(json.dumps(text[:MAX_PAGE_CHARS]))
    return json.loads(path.read_text())


page = await fetch(results[0]["url"], meter)
print(len(page), "chars:", page[:300])

9653 chars: Harriet Rebecca Lane Johnston (née Lane; May 9, 1830 – July 3, 1903) acted as First Lady of the United States during the administration of her uncle, President James Buchanan, from 1857 to 1861. She has been described as the first of the modern first ladies, being a diplomatic hostess, whose dress-s


## 2.2 Evidence Notes

No page ever enters a planner, worker or writer prompt. For each page read, an extractor
returns up to three **notes**: a one-sentence claim plus a quote copied from the page. A note
is kept only if its quote really occurs in the page text (after whitespace and case
normalization), so an invented quote is dropped before anything can cite it. Each note also
records which sub-question it was gathered for, so a worker sees only its own notes.

In [7]:
@dataclass
class Note:
    """One piece of evidence: claim, verbatim quote, source URL, owning sub-question."""

    id: int
    claim: str
    quote: str
    url: str
    topic: str


@dataclass
class EvidenceStore:
    """All notes of one research run, shared by every worker."""

    notes: list[Note] = field(default_factory=list)
    seen_urls: set[str] = field(default_factory=set)

    def render(self, topic: str | None = None) -> str:
        """Notes as numbered lines for a prompt, optionally only one sub-question's."""
        chosen = [n for n in self.notes if topic is None or n.topic == topic]
        return "\n".join(f'[{n.id}] {n.claim} (quote: "{n.quote}")' for n in chosen)


EXTRACT_SCHEMA = obj(notes={"type": "array", "items": obj(claim=STR, quote=STR)})


def squash(text: str) -> str:
    """Lower-case and collapse whitespace for quote matching."""
    return re.sub(r"\s+", " ", text).strip().lower()

In [8]:
async def read_page(
    question: str, url: str, store: EvidenceStore, meter: Meter
) -> list[Note]:
    """Extract notes for `question` from one page; keep notes whose quote is on it."""
    page = await fetch(url, meter)
    if len(page) < 200:
        return []
    reply = await llm(
        meter,
        "Extract up to 3 facts from the page that help answer the question. For "
        "each, give a one-sentence claim and a supporting quote copied exactly from "
        "the page (10-40 words). Return an empty list if the page does not help.",
        f"Question: {question}\nPage ({url}):\n{page}",
        EXTRACT_SCHEMA,
        effort="minimal",
    )
    new_notes = []
    for item in reply["notes"]:
        if squash(item["quote"]) in squash(page):
            note = Note(
                len(store.notes) + 1, item["claim"], item["quote"], url, question
            )
            store.notes.append(note)
            new_notes.append(note)
    return new_notes


store, meter = EvidenceStore(), Meter()
new_notes = await read_page(EXAMPLE["question"], results[0]["url"], store, meter)
print(store.render())

[1] The 15th First Lady of the United States is Harriet Rebecca Lane Johnston (Harriet Lane). (quote: "Harriet Rebecca Lane Johnston (née Lane; May 9, 1830 – July 3, 1903) acted as First Lady of the United States during the administration of her uncle, President James Buchanan, from 1857 to 1861.")
[2] Harriet Lane's mother's name was Jane Ann Buchanan Lane. (quote: "She was the youngest child of Elliott Tole Lane, a merchant, and Jane Ann Buchanan Lane.")


---

# 3) The Research Loop: One Worker

A worker researches **one question** with a short loop. Each step decides whether its notes
already answer the question; if not, it writes the next query, searches, and reads the top
unseen pages in parallel. The same function serves two roles:

- the **single-loop baseline**: the worker on the full FRAMES question, `BASELINE_STEPS`
  steps;
- a **deep-research worker**: the worker on one atomic sub-question, `WORKER_STEPS` steps,
  followed by a short sub-answer that cites its notes.

The decision prompt asks for a quote-backed fact for every hop. It explicitly forbids filling
hops from memory, the failure mode of a loop that only looks for "the single most useful
missing fact".

In [9]:
NEXT_SCHEMA = obj(done=BOOL, query=STR, reason=STR)


async def research_loop(
    question: str,
    store: EvidenceStore,
    meter: Meter,
    max_steps: int,
    log: Callable[[str], None] | None = None,
) -> None:
    """Search, read and take notes for `question` until done or out of steps."""
    for step in range(1, max_steps + 1):
        decision = await llm(
            meter,
            "You are researching a question with web search. Set done=true only if "
            "the notes contain a quoted fact for every step of the answer; never fill "
            "a step from memory. Otherwise write the next search query (keywords) for "
            "the most useful missing fact, using names already found in the notes.",
            f"Question: {question}\n"
            f"Notes so far:\n{store.render(question) or '(none)'}",
            NEXT_SCHEMA,
        )
        if decision["done"]:
            break
        results = await search(decision["query"], meter)
        urls = [r["url"] for r in results if r["url"] not in store.seen_urls]
        urls = urls[:READS_PER_STEP]
        store.seen_urls.update(urls)
        note_lists = await asyncio.gather(
            *(read_page(question, url, store, meter) for url in urls)
        )
        if log:
            found = sum(len(notes) for notes in note_lists)
            log(f"    step {step}: search {decision['query']!r} -> {found} notes")

In [10]:
SUBANSWER_SCHEMA = obj(answer=STR, found=BOOL)


@dataclass
class Finding:
    """A worker's result: sub-question, short answer, whether the notes support it."""

    question: str
    answer: str
    found: bool


async def worker(
    sub_question: str,
    store: EvidenceStore,
    meter: Meter,
    log: Callable[[str], None] | None = None,
) -> Finding:
    """Research one sub-question in its own context and answer it from its notes."""
    await research_loop(sub_question, store, meter, WORKER_STEPS, log)
    reply = await llm(
        meter,
        "Answer the question in one short sentence using only the notes, citing note "
        "ids in square brackets. If the notes do not answer it, set found=false and "
        "say what is missing.",
        f"Question: {sub_question}\nNotes:\n{store.render(sub_question) or '(none)'}",
        SUBANSWER_SCHEMA,
    )
    return Finding(sub_question, reply["answer"], reply["found"])


store, meter = EvidenceStore(), Meter()
finding = await worker(
    "Who was the 15th First Lady of the United States?", store, meter, log=print
)
print(finding)

    step 1: search '15th First Lady of the United States who' -> 4 notes


Finding(question='Who was the 15th First Lady of the United States?', answer='The 15th First Lady of the United States was Harriet Rebecca Lane Johnston [3].', found=True)


---

# 4) Deep Research: Plan, Parallel Workers, Gap Check

## 4.1 Planner

The planner decomposes the question into **atomic** sub-questions, each asking for one fact.
Two rules make the plan useful:

- **Answerable now.** A sub-question must not depend on an answer not yet found ("Who was the
  mother of [the 15th First Lady]?" has to wait until the First Lady is known). Dependent
  hops are left to later rounds, where the gap check writes them with the entity filled in.
- **Verify, don't assume.** A question's premises ("the second assassinated president") become
  sub-questions of their own. Nothing the answer rests on may come from the model's memory.

In [11]:
PLAN_SCHEMA = obj(sub_questions=STR_LIST)
PLANNER_SYSTEM = (
    "You plan web research for a multi-hop question. Break it into at most {num} "
    "atomic sub-questions, each asking for ONE fact that can be searched right now. "
    "Do not write sub-questions that depend on an answer not yet known; those come "
    "later. Turn every premise of the question (who is 'the 15th...', 'the "
    "second...', dates, rankings) into its own sub-question instead of assuming it."
)


async def plan(question: str, meter: Meter) -> list[str]:
    """Initial atomic, independent sub-questions for the question."""
    reply = await llm(
        meter,
        PLANNER_SYSTEM.format(num=MAX_SUBQUESTIONS),
        f"Question: {question}",
        PLAN_SCHEMA,
        effort="medium",
    )
    return reply["sub_questions"][:MAX_SUBQUESTIONS]


for sub_question in await plan(EXAMPLE["question"], Meter()):
    print("-", sub_question)

- Who was the 15th First Lady of the United States?
- What was the name of the mother of the 15th First Lady of the United States?
- Which U.S. president was the second to be assassinated?
- What was the maiden name of the mother of the second U.S. president to be assassinated?


## 4.2 Gap Check

After each round of workers, a reviewer reads the question, every finding and every note,
and answers three questions:

| Check | Example | Action |
|---|---|---|
| **Missing** | the mother of the First Lady is not yet known | follow-up with the entity filled in |
| **Conflicting** | two notes give different numberings of First Ladies | follow-up that settles the conflict |
| **Unverified** | a step of the reasoning rests on no note | follow-up that searches it |

It returns `ready=true` only when every step of the answer is backed by a note. Otherwise
its follow-up questions become the next round's sub-questions. This step is where the agent
recovers from a wrong first plan.

In [12]:
GAP_SCHEMA = obj(
    ready=BOOL, missing=STR_LIST, conflicts=STR_LIST, follow_up_questions=STR_LIST
)
GAP_SYSTEM = (
    "You review research progress on a multi-hop question. List the facts still "
    "missing, any conflicts between notes or findings, and any reasoning step not "
    "backed by a note. Set ready=true only if every step of the answer is supported "
    "by notes. Otherwise write at most {num} follow-up search questions that fill the "
    "gaps or settle the conflicts. Make each one atomic and concrete: substitute the "
    "names and values already found. Never repeat a sub-question already researched."
)


def render_findings(findings: list[Finding]) -> str:
    """Findings as lines for a prompt."""
    return "\n".join(
        f"- Q: {f.question}\n  A: {f.answer} ({'found' if f.found else 'NOT found'})"
        for f in findings
    )


async def gap_check(
    question: str, findings: list[Finding], store: EvidenceStore, meter: Meter
) -> dict:
    """Decide whether research is complete; if not, return follow-up questions."""
    return await llm(
        meter,
        GAP_SYSTEM.format(num=MAX_SUBQUESTIONS),
        f"Question: {question}\n\nFindings:\n{render_findings(findings)}\n\n"
        f"Notes:\n{store.render() or '(none)'}",
        GAP_SCHEMA,
        effort="medium",
    )

## 4.3 The Orchestrator

Each round runs its sub-questions concurrently (`asyncio.gather`). Every worker reads only
its own notes, which is the main benefit of multi-agent research: the worker for "who was
Garfield's mother" never wades through pages about First Ladies. The gap check and the writer
see everything. Research stops when the gap check says ready, when it proposes nothing new,
or after `MAX_ROUNDS`.

In [13]:
@dataclass
class Research:
    """Everything a deep-research run produced."""

    store: EvidenceStore
    findings: list[Finding]
    rounds: int


async def deep_research(question: str, meter: Meter, verbose: bool = False) -> Research:
    """Plan -> parallel workers -> gap check, repeated until ready or out of rounds."""
    log = print if verbose else None
    store, findings = EvidenceStore(), []
    sub_questions = await plan(question, meter)
    for round_num in range(1, MAX_ROUNDS + 1):
        if verbose:
            print(f"round {round_num}: {len(sub_questions)} sub-questions")
            for sub_question in sub_questions:
                print(f"  * {sub_question}")
        new = await asyncio.gather(
            *(worker(q, store, meter, log) for q in sub_questions)
        )
        findings += new
        if verbose:
            for f in new:
                print(f"  -> {f.answer} ({'found' if f.found else 'NOT found'})")
        review = await gap_check(question, findings, store, meter)
        if verbose:
            print(
                f"  gap check: ready={review['ready']} missing={review['missing']} "
                f"conflicts={review['conflicts']}"
            )
        asked = {f.question for f in findings}
        sub_questions = [q for q in review["follow_up_questions"] if q not in asked]
        if review["ready"] or not sub_questions:
            break
    return Research(store, findings, round_num)

## 4.4 Cited Answer and Citation Precision

The writer sees the findings and the numbered notes, never the pages. It must put note ids
after each claim it takes from them. If the notes miss a fact, it may fill the gap from its
own knowledge but must mark that sentence `[no source]`, so an incomplete search degrades to
a closed-book answer instead of a refusal.

**Citation precision** is the fraction of the report's claims that are supported by the notes
they cite, as judged by a model. A `[no source]` claim counts as unsupported, so precision
directly measures how much of the answer rests on evidence.

In [14]:
ANSWER_SCHEMA = obj(report=STR, final_answer=STR)
CHECK_SCHEMA = obj(claims={"type": "array", "items": obj(claim=STR, supported=BOOL)})


async def write_answer(
    question: str, store: EvidenceStore, meter: Meter, findings: str = ""
) -> dict:
    """Cited report and short final answer; unsourced claims are marked."""
    return await llm(
        meter,
        "Answer the question from the numbered notes. Write 2-6 sentences with the "
        "note ids in square brackets after every claim, e.g. [2][5]. If the notes miss "
        "a fact, use your own knowledge and end that sentence with [no source]. Then "
        "give the final answer as a short phrase.",
        f"Question: {question}\n{findings}\nNotes:\n{store.render() or '(none)'}",
        ANSWER_SCHEMA,
        effort="medium",
    )


async def citation_precision(report: str, store: EvidenceStore, meter: Meter) -> float:
    """Fraction of the report's claims supported by the notes they cite."""
    reply = await llm(
        meter,
        "Split the report into its factual claims. For each claim, decide whether the "
        "notes it cites (by id in square brackets) support it. A claim with no "
        "citation or marked [no source] is unsupported.",
        f"Report:\n{report}\n\nNotes:\n{store.render()}",
        CHECK_SCHEMA,
    )
    claims = reply["claims"]
    return sum(c["supported"] for c in claims) / max(1, len(claims))

---

# 5) Walkthrough

One FRAMES question end to end: the planner's sub-questions, every worker's searches, each
round's findings, and the gap check's verdict.

In [15]:
meter = Meter()
start = time.perf_counter()
research = await deep_research(EXAMPLE["question"], meter, verbose=True)
answer = await write_answer(
    EXAMPLE["question"], research.store, meter, render_findings(research.findings)
)
seconds = time.perf_counter() - start

print("\nREPORT:", answer["report"])
print("FINAL :", answer["final_answer"], "| GOLD:", EXAMPLE["gold"])
tokens = meter.input_tokens + meter.output_tokens
print(
    f"{research.rounds} rounds, {len(research.findings)} sub-questions, "
    f"{meter.searches} searches, {meter.fetches} pages, {len(research.store.notes)} "
    f"notes, {tokens:,} tokens, ${meter.dollars:.4f}, {seconds:.0f}s"
)
precision = await citation_precision(answer["report"], research.store, Meter())
print(f"citation precision: {precision:.0%}")

round 1: 4 sub-questions
  * Who was the 15th First Lady of the United States?
  * What was the first name of the mother of the 15th First Lady of the United States?
  * Who was the second U.S. President to be assassinated?
  * What was the mother's maiden name of the second U.S. President to be assassinated?


    step 1: search 'who was the 15th First Lady of the United States list of First Ladies 15th' -> 2 notes


    step 1: search '15th First Lady of the United States who was the 15th First Lady' -> 4 notes


    step 1: search 'James A. Garfield mother maiden name Eliza Ballou Garfield source' -> 2 notes


    step 1: search 'second U.S. president to be assassinated' -> 3 notes


    step 2: search 'Letitia Tyler mother name' -> 2 notes


... (20 outputs left out)

  -> Harriet Rebecca Lane Johnston served as Acting First Lady (Hostess) for President James Buchanan during his presidency (1857–1861) [22]. (found)
  -> found=false: the notes confirm Harriet Rebecca Lane Johnston acted as First Lady/hostess for James Buchanan [23][24] but do not provide her mother's given (first) name. (NOT found)
  -> Yes — the monument at her birthplace identifies her maiden name as Eliza Ballou [25]. (found)
  -> Wikidata lists Mehitable Ingalls as the mother of Eliza Ballou, confirming Ingalls as Eliza’s maternal maiden name rather than Eliza’s own maiden name [29][30][31]. (found)


  gap check: ready=False missing=['An authoritative source (WhiteHouse.gov, Library of Congress, or comparable) that explicitly lists who is the 15th person to serve as First Lady of the United States and confirms that person’s identity (to resolve whether the 15th First Lady is Harriet Rebecca Lane Johnston or Letitia Christian Tyler).', "An authoritative source that states the 15th First Lady’s mother’s given (first) name (i.e., confirmation that Harriet Rebecca Lane Johnston’s mother was named 'Jane Buchanan Lane' if Harriet is the 15th First Lady).", 'A single authoritative citation tying together: (a) the 15th First Lady = Harriet Rebecca Lane Johnston, and (b) her mother’s given name = Jane — so the correct first name to use in the final assembled name is unambiguous.'] conflicts=['Some notes state the 15th First Lady is Letitia Christian Tyler (notes [1], [12]) while others state Harriet Rebecca Lane Johnston served as First Lady/hostess for the 15th President James Buchanan (no


REPORT: The 15th First Lady (White House hostess) was Harriet Rebecca Lane Johnston, who served as hostess for President James Buchanan [3][15][22]. Harriet Rebecca Lane Johnston’s mother was named Jane Buchanan Lane [14]. The second U.S. President to be assassinated was James A. Garfield [9][10]. James A. Garfield’s mother’s maiden name was Ballou (Eliza Ballou Garfield) [18][25].
FINAL : Jane Ballou | GOLD: Jane Ballou
3 rounds, 12 sub-questions, 22 searches, 42 pages, 31 notes, 78,309 tokens, $0.0585, 202s


citation precision: 100%


**What the run shows.** The planner turned both premises ("the 15th First Lady", "the
second assassinated president") into sub-questions of their own. It also asked for the two
*mothers* in round 1 despite its instruction, so those workers had to resolve both hops
themselves. Two of the four round-1 findings were wrong: "Mary", which is the mother of
Letitia Tyler and came from a page that counts First Ladies differently, and "Ingalls", the
maiden name of Garfield's *grandmother*. A single loop would have answered from those
findings. The gap check flagged both as conflicts and misattributions, and round 2's targeted
follow-ups settled them: Jane Buchanan Lane, Eliza Ballou. Round 3 was wasted
over-verification. The reviewer kept asking for "authoritative" sources for facts that were
already quoted, so the run stopped on the round cap, not on `ready`.

---

# 6) Comparison on FRAMES

Four ways to answer the same 12 questions:

| Method | Search | Structure |
|---|---|---|
| **Closed-book** | none | the model answers from memory |
| **Single search** | 1 query = the question | read the top 2 pages, answer from their notes |
| **Single loop** | up to 6 sequential queries | one worker on the whole question |
| **Deep research** | plan + parallel workers + gap-check rounds | as built above |

Each method's answer is graded against the gold answer by a model. For the two research
methods, the citation precision of their reports is measured as well. All questions run
concurrently.

In [16]:
GRADE_SCHEMA = obj(correct=BOOL)


async def grade(question: str, gold: str, predicted: str) -> bool:
    """Model judgement of whether `predicted` means the same as `gold`."""
    reply = await llm(
        Meter(),
        "Reply correct=true if the predicted answer means the same as the gold answer "
        "(ignore phrasing and formatting), else false.",
        f"Question: {question}\nGold: {gold}\nPredicted: {predicted}",
        GRADE_SCHEMA,
        effort="minimal",
    )
    return reply["correct"]


async def closed_book(question: str, meter: Meter) -> tuple[str, float | None]:
    """Answer from the model's memory, no search."""
    reply = await llm(
        meter, "Answer the question.", question, ANSWER_SCHEMA, effort="medium"
    )
    return reply["final_answer"], None


async def single_search(question: str, meter: Meter) -> tuple[str, float | None]:
    """Search the question once, read the top pages, answer from their notes."""
    store = EvidenceStore()
    results = await search(question, meter)
    await asyncio.gather(
        *(read_page(question, r["url"], store, meter) for r in results[:READS_PER_STEP])
    )
    return (await write_answer(question, store, meter))["final_answer"], None

In [17]:
async def single_loop(question: str, meter: Meter) -> tuple[str, float | None]:
    """One sequential research loop on the whole question, then a cited answer."""
    store = EvidenceStore()
    await research_loop(question, store, meter, BASELINE_STEPS)
    answer = await write_answer(question, store, meter)
    precision = await citation_precision(answer["report"], store, Meter())
    return answer["final_answer"], precision


async def deep(question: str, meter: Meter) -> tuple[str, float | None]:
    """Deep research (plan, parallel workers, gap check), then a cited answer."""
    research = await deep_research(question, meter)
    answer = await write_answer(
        question, research.store, meter, render_findings(research.findings)
    )
    precision = await citation_precision(answer["report"], research.store, Meter())
    return answer["final_answer"], precision

In [18]:
Solver = Callable[[str, Meter], Awaitable[tuple[str, float | None]]]


async def evaluate(solve: Solver) -> dict:
    """Run `solve` on every question; return mean metrics and per-question marks."""

    async def one(q: dict) -> tuple[bool, Meter, float, float | None]:
        """Solve and grade one question."""
        meter, start = Meter(), time.perf_counter()
        predicted, precision = await solve(q["question"], meter)
        seconds = time.perf_counter() - start
        correct = await grade(q["question"], q["gold"], predicted)
        return correct, meter, seconds, precision

    runs = await asyncio.gather(*(one(q) for q in QUESTIONS))
    num = len(runs)
    precisions = [p for *_, p in runs if p is not None]
    return {
        "accuracy": sum(r[0] for r in runs) / num,
        "searches": sum(r[1].searches for r in runs) / num,
        "pages": sum(r[1].fetches for r in runs) / num,
        "dollars": sum(r[1].dollars for r in runs) / num,
        "seconds": sum(r[2] for r in runs) / num,
        "precision": sum(precisions) / len(precisions) if precisions else None,
        "correct": [r[0] for r in runs],
    }

In [19]:
start = time.perf_counter()
methods = [
    ("closed-book", closed_book),
    ("single search", single_search),
    ("single loop", single_loop),
    ("deep research", deep),
]
table = dict(
    zip(
        [n for n, _ in methods],
        await asyncio.gather(*(evaluate(solve) for _, solve in methods)),
    )
)
print(f"all methods: {time.perf_counter() - start:.0f}s wall clock\n")

header = f"{'method':<15}{'acc':>6}{'searches':>10}{'pages':>7}{'$/q':>9}{'sec/q':>7}"
print(header + f"{'cite prec':>11}  per question")
for name, row in table.items():
    marks = "".join("✓" if ok else "·" for ok in row["correct"])
    cite = f"{row['precision']:.0%}" if row["precision"] is not None else "-"
    print(
        f"{name:<15}{row['accuracy']:>6.0%}{row['searches']:>10.1f}{row['pages']:>7.1f}"
        f"{row['dollars']:>9.4f}{row['seconds']:>7.0f}{cite:>11}  {marks}"
    )
await HTTP.aclose()

all methods: 574s wall clock

method            acc  searches  pages      $/q  sec/q  cite prec  per question
closed-book       50%       0.0    0.0   0.0030     16          -  ✓·✓✓✓···✓·✓·
single search     58%       1.0    1.9   0.0041     27          -  ✓✓✓✓✓·✓···✓·
single loop       50%       4.8    8.9   0.0109     81        68%  ✓·✓✓✓··✓··✓·
deep research     67%      19.1   31.1   0.0472    203        85%  ✓·✓✓✓·✓✓✓·✓·


---

# 7) Practical Notes

**Reading the table.** In this run, deep research answered 8 of 12 questions, against 6 to
7 for the other methods. Its citation precision was 85%, against 68% for the single loop, at
about 4x the single loop's cost and 2.5x its time. With 12 questions, one question is 8 points of accuracy, so only large
gaps are meaningful. The more robust signals are citation precision, which shows how much of
each answer is grounded in fetched evidence rather than memory, and the cost columns, which
show what the extra depth buys. FRAMES questions were written in 2024. A strong model already
knows many answers closed-book, and research mostly pays off on the questions it gets wrong
or would have to guess.

| Knob | Here | Production | Effect |
|---|---|---|---|
| Sub-questions per round | 4 | 3 to 20 (scaled to difficulty) | breadth of each round |
| Worker steps | 3 | 10 to 100+ tool calls | depth per sub-question |
| Rounds | 3 | until budget or ready | recovery from bad plans |
| Pages per search | 2 | 5 to 20, reranked | recall vs. tokens |
| Worker model | gpt-5-mini | small/fast model | cost is dominated by page reading |
| Planner / reviewer model | gpt-5-mini, medium effort | strongest model | plan quality caps everything |

**Failure modes.**

- *Over-decomposition*: a simple lookup split into many sub-questions spends 10x the tokens.
  Anthropic's system scales the number of subagents to the estimated difficulty for this
  reason. A planner instruction like "one sub-question for a simple fact" is the cheap fix.
- *Over-verification*: the reviewer keeps asking for "more authoritative" confirmation of
  facts that are already quoted, as in the walkthrough's third round. Tell it that one quoted
  page is enough unless notes conflict, and cap the rounds.
- *Premature ready*: the gap check accepts a plausible but unsupported step. Requiring a
  note id for every step of the answer, as done here, is the main defense.
- *Follow-up loops*: the reviewer keeps proposing reworded versions of the same question.
  Deduplicate against what was already asked, and cap the rounds.
- *Contamination*: benchmark questions appear online with answers; block those sources, as
  `BLOCKED_DOMAINS` does.
- *Prompt injection from pages*: page text is untrusted. Here pages only reach the extractor,
  which returns claims and checked quotes. The planner, reviewer and writer never see raw
  pages.
- *Blocked fetches*: many sites refuse automated clients. Empty pages silently weaken every
  method that reads them.

**What production systems add.** A model trained with RL on browsing tasks instead of
prompted steps (OpenAI deep research, Kimi-Researcher), which writes better queries and makes
better stopping decisions. Source ranking by reliability rather than search position. And
long-horizon memory, so a research run of an hour does not overflow the orchestrator's
context.

---

# 8) Conclusion

| Step | What it does | Design choice |
|---|---|---|
| Plan | question -> atomic, assumption-free sub-questions | dependent hops deferred to later rounds |
| Work | one short search/read loop per sub-question | parallel; each worker sees only its own notes |
| Read | page -> notes with claim + quote | quotes verified against the page text |
| Gap check | missing / conflicting / unverified -> follow-ups | entities from earlier rounds filled in |
| Answer | cited report from notes only | `[no source]` marks memory-only claims |
| Check | citation precision | model judges each claim against its quotes |

**References**

- OpenAI, *Introducing deep research* (2025).
- Anthropic, *How we built our multi-agent research system* (2025).
- Krishna et al., *Fact, Fetch, and Reason: A Unified Evaluation of Retrieval-Augmented
  Generation* (FRAMES), 2024. arXiv:2409.12941.
- Wei et al., *BrowseComp: A Simple Yet Challenging Benchmark for Browsing Agents*, 2025.
  arXiv:2504.12516.
- `langchain-ai/open_deep_research`, `dzhng/deep-research`, `assafelovic/gpt-researcher`.